I. Loads the loading script for the MultiEURLEX dataset. 

In [ ]:
import requests
script = requests.get('https://huggingface.co/datasets/coastalcph/multi_eurlex/resolve/main/multi_eurlex.py')
print(script.text)

1. This cell loads all the necessary components for the evaluation. First, it loads the MultiEURLEX test set containing 5,000 legal documents with all 7,390 possible labels. Then it downloads the EuroVoc descriptor file from GitHub, which maps label IDs to their English text descriptions. Next, it extracts the label IDs from the dataset and converts them to readable text using the descriptor file. After that, it loads the multilingual E5-small embedding model. Finally, it embeds all 7,390 label descriptions into 384-dimensional vectors.

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import requests
from tqdm import tqdm
import pandas as pd

dataset = load_dataset('coastalcph/multi_eurlex', 'de', split='test',
                       label_level='all_levels', trust_remote_code=True)

classlabel = dataset.features["labels"].feature
print(f"Number of labels: {len(classlabel.names)}")

url = "https://raw.githubusercontent.com/nlpaueb/multi-eurlex/master/data/eurovoc_descriptors.json"
eurovoc_concepts = requests.get(url).json()
label_ids = classlabel.names
label_descriptors = ["passage: " + eurovoc_concepts[label_id]['en'] for label_id in label_ids]

model = SentenceTransformer('intfloat/multilingual-e5-small')
label_embeddings = model.encode(label_descriptors, show_progress_bar=True, batch_size=32)

2. This cell defines three standard evaluation metrics used in information retrieval and multi-label classification. The precision at k function calculates what fraction of the top k retrieved labels are actually correct by dividing the number of correct labels in the top k by k. The recall at k function calculates what fraction of the true labels were found in the top k predictions by dividing the number of correct labels found by the total number of true labels. The NDCG at k function computes normalized discounted cumulative gain, which gives higher scores when correct labels appear earlier in the ranking, with labels at position 1 weighted more than those at position 10. NDCG also compares the actual ranking to an ideal ranking where all true labels would be ranked first. All three metrics return values between 0 and 1, where higher is better.

In [ ]:
# Evaluation metrics functions
def precision_at_k(y_true, y_pred, k):
    """Precision@k: fraction of top-k predictions that are correct"""
    top_k = y_pred[:k]
    relevant = sum(1 for label in top_k if label in y_true)
    return relevant / k

def recall_at_k(y_true, y_pred, k):
    """Recall@k: fraction of true labels found in top-k"""
    if len(y_true) == 0:
        return 0.0
    top_k = y_pred[:k]
    relevant = sum(1 for label in top_k if label in y_true)
    return relevant / len(y_true)

def ndcg_at_k(y_true, y_pred, k):
    """NDCG@k: Normalized Discounted Cumulative Gain"""
    top_k = y_pred[:k]
    dcg = sum((1 if label in y_true else 0) / np.log2(i + 2) 
              for i, label in enumerate(top_k))
    
    # Ideal DCG (if all true labels were ranked first)
    ideal_k = min(len(y_true), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_k))
    
    return dcg / idcg if idcg > 0 else 0.0

print("Metric functions defined ✓")

3. This cell processes every document in the 5,000 document test set to calculate retrieval performance. First, all 5,000 documents are encoded in batches of 32 using the same embedding model, with each document text prepended with the prefix "query: " as required by the E5 model. Batch encoding is more computationally efficient than encoding documents one at a time, as it allows the model to process multiple documents in parallel on the GPU or CPU. The resulting document embeddings are stored in memory as a matrix before evaluation begins. For each document, the cell then extracts the true labels that human annotators assigned to that document and computes cosine similarity between the document embedding and all 7,390 label embeddings. The labels are ranked from highest to lowest similarity score, and for each k value in the list of 5, 10, 20, and 50, the cell calculates precision, recall, and NDCG using the top k predicted labels compared to the true labels. All metric scores are stored in lists so that averages and standard deviations can be computed later.

In [ ]:
k_values = [5, 10, 20, 50, 100]
results = {
    'precision': {k: [] for k in k_values},
    'recall': {k: [] for k in k_values},
    'ndcg': {k: [] for k in k_values}
}

# Pre-encode all documents in batches
texts = ["query: " + doc['text'] for doc in dataset]
doc_embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

# Evaluate
for idx, doc in enumerate(tqdm(dataset, desc="Evaluating")):
    true_labels = set(doc['labels'])
    if len(true_labels) == 0:
        continue
    similarities = cosine_similarity([doc_embeddings[idx]], label_embeddings)[0]
    ranked_predictions = np.argsort(similarities)[::-1]
    for k in k_values:
        results['precision'][k].append(precision_at_k(true_labels, ranked_predictions, k))
        results['recall'][k].append(recall_at_k(true_labels, ranked_predictions, k))
        results['ndcg'][k].append(ndcg_at_k(true_labels, ranked_predictions, k))

4. This cell takes all the metric scores collected during evaluation and computes summary statistics. For each metric and each k value, it calculates the mean score across all 5,000 documents and the standard deviation to show how much variation exists. The results are printed in a readable format showing precision, recall, and NDCG for k equals 5, 10, 20, and 50, each with standard deviation. Then it creates a pandas dataframe to display all results in a clean table format with columns for k, precision, recall, and NDCG. This table provides a quick overview of model performance at different retrieval depths, making it easy to see that recall increases with larger k values while precision typically decreases.

In [ ]:
for metric_name, metric_data in results.items():
    for k in k_values:
        scores = metric_data[k]
        mean = np.mean(scores)
        std = np.std(scores)

summary_data = []
for k in k_values:
    summary_data.append({
        'k': k,
        'Precision': f"{np.mean(results['precision'][k]):.4f} ± {np.std(results['precision'][k]):.4f}",
        'Recall': f"{np.mean(results['recall'][k]):.4f} ± {np.std(results['recall'][k]):.4f}",
        'NDCG': f"{np.mean(results['ndcg'][k]):.4f} ± {np.std(results['ndcg'][k]):.4f}"
    })
summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

5. This cell saves the complete evaluation results to a JSON file for later analysis and comparison. It stores metadata about the experiment including the model name, dataset, language, number of documents and number of labels, alongside the full metric results. For each combination of metric and k value, it saves both the mean score across all documents and the standard deviation, as well as the individual per-document scores. Saving the individual scores allows for statistical comparisons between models later, such as computing confidence intervals or running significance tests. The file is named to reflect the model and language used, making it easy to keep results from different experiments organised and preventing accidental overwriting when running the pipeline on other models or languages.

In [ ]:
import json

results_to_save = {
    'model': 'multilingual-e5-small',
    'dataset': 'MultiEURLEX',
    'language': 'German (EN labels)',
    'num_documents': len(dataset),
    'num_labels': len(label_descriptors),
    'metrics': {
        metric_name: {
            k: {
                'mean': float(np.mean(scores)),
                'std': float(np.std(scores)),
                'values': [float(x) for x in scores]
            }
            for k, scores in metric_data.items()
        }
        for metric_name, metric_data in results.items()
    }
}

with open('results_e5_german_enlabels.json', 'w') as f:
    json.dump(results_to_save, f, indent=2)